# Binary Classification with Pytorch 
* Bankruptcy Dataset

In [1]:
# torch imports
import torch 
import torch.nn as nn
from torchsummary import summary

In [2]:
#  sklearn imports 
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

In [3]:
# General Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.chdir("D:\Machine Learning\Datasets")
import warnings
warnings.filterwarnings('ignore')

In [4]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")


Is CUDA available? False
Number of GPUs: 0


In [5]:
brupt = pd.read_excel("bankruptcy.xlsx", sheet_name='data',index_col=0)
X=brupt.drop(['D', 'YR'], axis=1).values
y=brupt['D'].values

In [6]:
X.shape

(132, 24)

In [7]:
scaler = MinMaxScaler()
X_train, X_test, y_train, y_test = train_test_split(X,y, 
                                                    test_size=0.3,
                                                    random_state=25,
                                                    stratify=y)
X_scl_trn = scaler.fit_transform(X_train)
X_scl_tst = scaler.transform(X_test)

In [8]:
X_torch = torch.from_numpy(X_scl_trn)
y_torch = torch.from_numpy(y_train)
print(X_torch.size())
print(y_torch.size())

torch.Size([92, 24])
torch.Size([92])


### Model Defination


In [9]:
torch.manual_seed(25)
# model = nn.Sequential(nn.Linear(in_features=X_scl_trn.shape[1],out_features=1)) # Accuracy → 0.7, log_loss → 0.564
model = nn.Sequential(nn.Linear(in_features=X_scl_trn.shape[1], out_features=18),
                      nn.ReLU(),
                      nn.Linear(in_features=18, out_features=7),
                      nn.ReLU(),
                      nn.Linear(in_features=7, out_features=3),
                      nn.ReLU(),
                      nn.Linear(in_features=3,out_features=1))

In [10]:
summary(model, (1,X_scl_trn.shape[1]))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                [-1, 1, 18]             450
              ReLU-2                [-1, 1, 18]               0
            Linear-3                 [-1, 1, 7]             133
              ReLU-4                 [-1, 1, 7]               0
            Linear-5                 [-1, 1, 3]              24
              ReLU-6                 [-1, 1, 3]               0
            Linear-7                 [-1, 1, 1]               4
Total params: 611
Trainable params: 611
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [11]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.05)
optimizer

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.05
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

### Generating predictions with default weights

In [12]:
y_pred = model(X_torch.float()) # to generate with default weights
y_pred[:3]

tensor([[-0.4664],
        [-0.4697],
        [-0.4707]], grad_fn=<SliceBackward0>)

In [13]:

print(y_torch.shape)

torch.Size([92])


In [14]:
y_torch = y_torch.unsqueeze(1) # changes the shape to match with the y_pred
print(y_torch.shape)
print(y_pred.shape)

torch.Size([92, 1])
torch.Size([92, 1])


### Initial Log_loss

In [15]:
loss = criterion(y_pred,y_torch.float())
loss

tensor(0.7206, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)

* .fit() → runs the loop on his own but here we have to run the epoch loop

In [16]:
for epoch in np.arange(0,1000):
    y_pred_prob = model(X_torch.float()) # Inferencing of X
    loss = criterion(y_pred_prob, y_torch.float())
    if epoch % 100 == 0:
        print('Epoch', epoch+1, 'Loss:', loss.item())
    optimizer.zero_grad() # all gradients are equaled to 0
    loss.backward()
    optimizer.step()


Epoch 1 Loss: 0.7206125855445862
Epoch 101 Loss: 0.6937741637229919
Epoch 201 Loss: 0.6922138929367065
Epoch 301 Loss: 0.6914511322975159
Epoch 401 Loss: 0.6903999447822571
Epoch 501 Loss: 0.6884443759918213
Epoch 601 Loss: 0.6851511001586914
Epoch 701 Loss: 0.678617537021637
Epoch 801 Loss: 0.6628679037094116
Epoch 901 Loss: 0.6170055866241455


## Training Set Log_loss

In [17]:
loss

tensor(0.5295, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)

In [18]:
X_torch_test = torch.from_numpy(X_scl_tst)
type(X_torch_test)

torch.Tensor

In [19]:
lin_output = model(X_torch_test.float())
type(lin_output)

torch.Tensor

In [20]:
np_out = lin_output.detach().numpy() # converts torch object to numpy
y_pred_prob =  1 / (1 + np.exp(-np_out))
y_pred_prob

array([[0.44918534],
       [0.67773676],
       [0.41853425],
       [0.36322528],
       [0.6740161 ],
       [0.36450732],
       [0.6006578 ],
       [0.67555255],
       [0.49324352],
       [0.7763044 ],
       [0.84018385],
       [0.5689584 ],
       [0.7190396 ],
       [0.36322528],
       [0.50023806],
       [0.46855587],
       [0.56338114],
       [0.55594856],
       [0.6344796 ],
       [0.5353673 ],
       [0.53023916],
       [0.6917931 ],
       [0.7428777 ],
       [0.5559378 ],
       [0.53416777],
       [0.6198235 ],
       [0.7006975 ],
       [0.7110978 ],
       [0.7923499 ],
       [0.51013064],
       [0.41125223],
       [0.5136846 ],
       [0.36322528],
       [0.72480214],
       [0.5039999 ],
       [0.41560978],
       [0.68595344],
       [0.38886753],
       [0.36322528],
       [0.36322528]], dtype=float32)

In [21]:
y_pred_prob.shape

(40, 1)

In [22]:
y_pred_prob = y_pred_prob.reshape(y_test.shape[0])
y_pred_prob.shape

(40,)

In [23]:
y_pred = np.where(y_pred_prob >= 0.5,1,0)
y_pred[:5]

array([0, 1, 0, 0, 1])

#  Test Accuracy Score

In [24]:
print(accuracy_score(y_test, y_pred))

0.675


In [25]:
log_loss(y_test, y_pred_prob)

0.6171060510759749